# Neuro-Symbolic Self-Verification Evidence

This notebook exercises SC-NeuroCore's neuro-symbolic self-verification trace: checked obligations, stable result digests, tamper detection, symbol-score validation, and trace-only symbolic evidence.

## Evidence Boundary

This notebook verifies internal consistency of generated neuro-symbolic inference artifacts only. It does not prove that the symbolic interpretation is externally true, clinically valid, physically grounded, or complete. External validation requires task-specific labels, independent datasets, domain review, and calibrated deployment evidence.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

from sc_neurocore.neuro_symbolic import (
    NeuroSymbolicPredictiveAgent,
    NeuroSymbolicSelfVerifier,
    PredictiveAgentConfig,
    SCErrorSignature,
    VerificationStatus,
    build_self_verification_trace,
)
from sc_neurocore.neuro_symbolic.agent import HybridInferenceResult

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

observation = np.array([0.25, -0.2, 0.1, -0.1], dtype=np.float32)
agent = NeuroSymbolicPredictiveAgent(
    PredictiveAgentConfig(
        input_dim=4,
        hidden_dim=2,
        symbols=("left", "right", "rest"),
        seed=3,
    )
)
result = agent.observe(observation, top_k=2)
trace = build_self_verification_trace(result, observation=observation)
pass_summary = {
    "schema_version": trace.schema_version,
    "passed": trace.passed,
    "digest_length": len(trace.result_digest),
    "reasoning_steps": trace.reasoning_steps,
    "top_symbols": list(trace.top_symbols),
    "sc_popcount": trace.sc_popcount,
    "sc_normalised_popcount": trace.sc_normalised_popcount,
    "obligations": [item.to_dict() for item in trace.obligations],
}
assert pass_summary["passed"] is True
assert pass_summary["digest_length"] == 64
pass_summary

In [ ]:
verifier = NeuroSymbolicSelfVerifier()
first = verifier.verify_result(result, observation=observation)
second = verifier.verify_result(result, observation=observation)
determinism_summary = {
    "same_digest": first.result_digest == second.result_digest,
    "same_payload": first.to_dict() == second.to_dict(),
}
assert determinism_summary["same_digest"] is True
assert determinism_summary["same_payload"] is True
determinism_summary

In [ ]:
tampered_signature = HybridInferenceResult(
    prediction=result.prediction,
    error=result.error,
    signature=SCErrorSignature(
        xor_bits=tuple(0 for _ in result.signature.xor_bits),
        popcount=0,
        normalised_popcount=0.0,
        mean_abs_error=result.signature.mean_abs_error,
    ),
    symbol_scores=result.symbol_scores,
    trace=result.trace,
    learned_error=result.learned_error,
)
tamper_trace = verifier.verify_result(tampered_signature, observation=observation)
tamper_summary = {
    "passed": tamper_trace.passed,
    "failed_obligations": list(tamper_trace.failed_obligations),
    "signature_obligation": next(
        item.to_dict()
        for item in tamper_trace.obligations
        if item.name == "sc_signature_consistency"
    ),
}
assert tamper_trace.passed is False
assert "sc_signature_consistency" in tamper_trace.failed_obligations
tamper_summary

In [ ]:
unsorted_symbols = HybridInferenceResult(
    prediction=result.prediction,
    error=result.error,
    signature=result.signature,
    symbol_scores=(("low", -0.5), ("high", 0.5)),
    trace=result.trace,
    learned_error=result.learned_error,
)
symbol_trace = verifier.verify_result(unsorted_symbols, observation=observation)
symbol_summary = {
    "passed": symbol_trace.passed,
    "failed_obligations": list(symbol_trace.failed_obligations),
    "symbol_obligation": next(
        item.to_dict()
        for item in symbol_trace.obligations
        if item.name == "symbol_score_ordering"
    ),
}
assert symbol_trace.passed is False
assert "symbol_score_ordering" in symbol_trace.failed_obligations
symbol_summary

In [ ]:
trace_only = verifier.verify_trace_only(
    result.trace,
    symbol_scores=result.symbol_scores,
    signature=result.signature,
)
trace_only_summary = {
    "passed": trace_only.passed,
    "reasoning_steps": trace_only.reasoning_steps,
    "top_symbols": list(trace_only.top_symbols),
    "sc_popcount": trace_only.sc_popcount,
    "obligation_names": [item.name for item in trace_only.obligations],
}
assert trace_only_summary["reasoning_steps"] == result.trace.length
assert trace_only_summary["sc_popcount"] == result.signature.popcount
trace_only_summary

In [ ]:
try:
    verifier.verify_result(result, observation=np.array([[1.0, 2.0]], dtype=np.float32))
except ValueError as exc:
    shape_refusal = str(exc)
else:
    raise AssertionError("non-vector observation was accepted")

try:
    verifier.verify_result(result, observation=np.array([1.0, np.nan, 2.0, 3.0]))
except ValueError as exc:
    finite_refusal = str(exc)
else:
    raise AssertionError("non-finite observation was accepted")

guardrail_summary = {
    "shape_refusal": shape_refusal,
    "finite_refusal": finite_refusal,
}
assert "one-dimensional" in shape_refusal
assert "finite" in finite_refusal
guardrail_summary

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.neuro-symbolic-self-verification-evidence.v1",
    "pass_summary": pass_summary,
    "determinism_summary": determinism_summary,
    "tamper_summary": tamper_summary,
    "symbol_summary": symbol_summary,
    "trace_only_summary": trace_only_summary,
    "guardrails": guardrail_summary,
    "evidence_boundary": "Internal consistency checks only; no external semantic-truth, clinical, physical, or deployment-validity claim.",
}
manifest